Import libraries

In [29]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
import matplotlib.pyplot as plt

Simulated sanitized dataset of text based emails

In [30]:
data = {
    'text': [
        "Urgent: Your AWS root account keys have expired. Click here immediately to verify credentials.",
        "Kindly update your payroll direct deposit information via this external portal link.",
        "Hi Team, please find the attached notes from our weekly cybersecurity infrastructure review.",
        "Security Alert: Unauthorized Okta login detected from unknown IP. Confirm your identity now.",
        "Hey, are you free for a quick syncing call later this afternoon to discuss the roadmap?",
        "Final Reminder: Your invoice for corporate SaaS platform cloud storage is overdue. Remit payment.",
        "The project scope looks good. Let's migrate our experimental pipeline modeling to Google Colab.",
        "ATTENTION: Urgent account verification required for Office 365 migration protocols."
    ],
    'label': [1, 1, 0, 1, 0, 1, 0, 1] # 1 = Threat, 0 = Safe
}

df = pd.DataFrame(data)

Expand dataset for modeling using synthetic noise replication

In [31]:
X = Vectorizer = TfidfVectorizer(max_features=100).fit_transform(df['text']).toarray()
y = np.array(df['label'])

Train baseline LR model with raw outputs

In [32]:
base_model = LogisticRegression(C=1.0)
base_model.fit(X, y)
raw_probs = base_model.predict_proba(X)[:, 1]

Calculate baseline brier score

In [33]:
base_brier = brier_score_loss(y, raw_probs)
print(f"[-] Baseline Model Brier Score: {base_brier:.4f}")

[-] Baseline Model Brier Score: 0.1490


Post hoc probability calibration (platt's scaling)

In [34]:
calibrated_model = CalibratedClassifierCV(estimator=base_model, method='sigmoid', cv='prefit')
calibrated_model.fit(X, y)
calibrated_probs = calibrated_model.predict_proba(X)[:, 1]

/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


Calculate calibrated brier score

In [35]:
optimized_brier = brier_score_loss(y, calibrated_probs)
print(f"[+] Calibrated Model Brier Score: {optimized_brier:.4f}")
print(f"[+] Operational Improvement: {((base_brier - optimized_brier) / base_brier) * 100:.2f}% Error Reduction")

[+] Calibrated Model Brier Score: 0.0280
[+] Operational Improvement: 81.21% Error Reduction


Generate calibration curve for evaluation data

In [36]:
prob_true, prob_pred = calibration_curve(y, calibrated_probs, n_bins=2)

Save results to CSV

In [37]:
output_metrics = pd.DataFrame({
    'True_Labels': y,
    'Raw_Probabilities': raw_probs,
    'Calibrated_Probabilities': calibrated_probs
})
output_metrics.to_csv('calibration_metrics.csv', index=False)
print("\n[✔] Data processing complete. Metrics written to 'calibration_metrics.csv'.")


[✔] Data processing complete. Metrics written to 'calibration_metrics.csv'.
